# Exercise 6.5 — Cross-Entropy Benchmarking Decay

**Chapter 6: Applications** &nbsp;|&nbsp; *Exercises*

---

## Motivation

**Cross-entropy benchmarking** (XEB) asks a different question from randomized benchmarking.  RB twirls the error channel and extracts a single average noise parameter, discarding everything specific to the circuit.  XEB instead tests whether one particular, computationally hard circuit was executed faithfully, by comparing sampled bitstrings against ideal probabilities computed classically.

The exercise shows that under a global depolarizing model the XEB score collapses to a strikingly simple form, $F_{\mathrm{XEB}} = (1-\varepsilon)^G$.  The entire circuit-specific structure drops out and leaves a bare product of per-gate survival factors.  This is what makes the metric readable, and it also fixes the regime in which the protocol can work at all: the fidelity decays exponentially in the gate count, so $G \sim 1/\varepsilon$ is the point beyond which the signal is no longer statistically resolvable.

## Preliminaries / Toolbox

**1. Linear XEB fidelity:** For $N$ qubits, $F_{\mathrm{XEB}} = 2^N\sum_x P_{\mathrm{ideal}}(x)P_{\mathrm{exp}}(x) - 1$.  A perfect device gives $F_{\mathrm{XEB}} \to 1$ for anti-concentrated circuits, and fully depolarized output gives $F_{\mathrm{XEB}} = 0$.

**2. Porter–Thomas distribution:** For circuits deep enough to anti-concentrate, the output probabilities follow $\rho(p) = 2^N e^{-2^N p}$, an exponential density with mean $\mu = 2^{-N}$ and second moment $\langle p^2\rangle = 2/4^N$.  This is the large-$D$ limit of the $\mathrm{Beta}(1, D-1)$ overlap distribution of Chapter 3.

**3. The collision probability:** The combination that appears in $F_{\mathrm{XEB}}$ is $2^N\sum_x P_{\mathrm{ideal}}(x)^2 \approx 2$.  The factor of two is the signature of Porter–Thomas statistics, and it is what normalizes the metric to unity.

**4. Global depolarizing model:** The cumulative effect of $G$ imperfect gates is modeled as $\hat{\rho}_{\mathrm{exp}} = (1-\varepsilon)^G\ket{\psi_{\mathrm{ideal}}}\!\bra{\psi_{\mathrm{ideal}}} + \bigl(1-(1-\varepsilon)^G\bigr)\hat{I}/2^N$, a mixture of the ideal state with uniform noise.

## Exercise Statement

A random circuit on $N$ qubits applies $G$ entangling gates, each with error probability $\varepsilon$.  Model the cumulative effect as global depolarization of the ideal output,
$$\hat{\rho}_{\mathrm{exp}} = (1-\varepsilon)^G\ket{\psi_{\mathrm{ideal}}}\!\bra{\psi_{\mathrm{ideal}}} + \bigl(1-(1-\varepsilon)^G\bigr)\,\hat{I}/2^N .$$

**(a)** Show that the linear cross-entropy fidelity $F_{\mathrm{XEB}} = 2^N\sum_x P_{\mathrm{ideal}}(x)P_{\mathrm{exp}}(x) - 1$ equals $(1-\varepsilon)^G$.

**(b)** For $\varepsilon = 10^{-3}$, at what gate count does $F_{\mathrm{XEB}}$ fall to $1/e$?

## Solution

### Part (a): the depolarizing model collapses the score to a product

Write $a = (1-\varepsilon)^G$ for the surviving weight of the ideal state.  Measuring $\hat{\rho}_{\mathrm{exp}}$ in the computational basis gives
$$P_{\mathrm{exp}}(x) = \bra{x}\hat{\rho}_{\mathrm{exp}}\ket{x} = a\,P_{\mathrm{ideal}}(x) + (1-a)\,2^{-N},$$
since the maximally mixed component contributes $2^{-N}$ to every bitstring.  Substituting into the definition,
$$F_{\mathrm{XEB}} = 2^N\sum_x P_{\mathrm{ideal}}(x)\left[a\,P_{\mathrm{ideal}}(x) + (1-a)2^{-N}\right] - 1,$$
and splitting the sum,
$$F_{\mathrm{XEB}} = a\,\underbrace{2^N\sum_x P_{\mathrm{ideal}}(x)^2}_{\text{collision term}} + (1-a)\,\underbrace{\sum_x P_{\mathrm{ideal}}(x)}_{=\,1} - 1 .$$

The second term used $2^N \cdot 2^{-N} = 1$ and normalization.  The first term is where anti-concentration enters.  For Porter–Thomas output,
$$\sum_x P_{\mathrm{ideal}}(x)^2 \approx 2^N\langle p^2\rangle = 2^N\cdot\frac{2}{4^N} = \frac{2}{2^N} \quad\Longrightarrow\quad 2^N\sum_x P_{\mathrm{ideal}}(x)^2 \approx 2 .$$

Therefore
$$F_{\mathrm{XEB}} \approx 2a + (1-a) - 1 = a,$$
$$\boxed{F_{\mathrm{XEB}} \approx (1-\varepsilon)^G .}$$

The two limits quoted in the text follow at once.  A perfect device has $a=1$ and $F_{\mathrm{XEB}} \to 1$; fully depolarized output has $a=0$ and $F_{\mathrm{XEB}} = 0$.

**Finite-size remark.**  For an exactly Haar-random ideal state the collision term is not quite 2.  With $D=2^N$ and $p\sim\mathrm{Beta}(1,D-1)$, the exact mean is $\langle p^2\rangle = 2/[D(D+1)]$, so
$$2^N\sum_x P_{\mathrm{ideal}}(x)^2 = D^2\cdot\frac{2}{D(D+1)} = \frac{2D}{D+1} \quad\Longrightarrow\quad F_{\mathrm{XEB}} = a\,\frac{D-1}{D+1}.$$
This approaches $a$ as $D\to\infty$ and is the finite-size correction the text alludes to.  The numerics below show both.

### Part (b): the $1/e$ point

Since $\varepsilon \ll 1$,
$$(1-\varepsilon)^G = e^{G\ln(1-\varepsilon)} \approx e^{-\varepsilon G},$$
so the fidelity falls to $1/e$ when $\varepsilon G \approx 1$:
$$\boxed{G \approx \frac{1}{\varepsilon} = 10^3 \text{ gates.}}$$

Keeping the logarithm exactly gives $G = -1/\ln(1-\varepsilon) = 999.5$, which rounds to the same $10^3$.  A circuit of this size sits at the edge of statistically resolvable fidelity, which is why random-circuit-sampling experiments operate near $G\sim 1/\varepsilon$.

---
## Symbolic Verification (SymPy)

In [ ]:
import sympy as sp

p, D, a, eps, G = sp.symbols('p D a varepsilon G', positive=True)

# Porter-Thomas density rho(p) = D e^{-D p} on p in [0, oo)
rho = D * sp.exp(-D * p)
print('Porter-Thomas density rho(p) =', rho)
print('  normalization  int rho dp =', sp.integrate(rho, (p, 0, sp.oo)))
print('  mean           <p>        =', sp.integrate(p * rho, (p, 0, sp.oo)))
print('  second moment  <p^2>      =', sp.integrate(p**2 * rho, (p, 0, sp.oo)))

# Collision term: 2^N sum_x P^2 = D * (D * <p^2>)
collision = D * (D * sp.integrate(p**2 * rho, (p, 0, sp.oo)))
print('\ncollision term 2^N sum_x P_ideal^2 =', sp.simplify(collision))

In [ ]:
# Part (a): assemble F_XEB with the collision term left symbolic
M2 = sp.Symbol('M_2')   # stands for 2^N sum_x P_ideal^2

F_XEB = a * M2 + (1 - a) * 1 - 1
print('F_XEB = a*M2 + (1-a) - 1 =', sp.simplify(F_XEB))

# Porter-Thomas value M2 = 2
print('  at M2 = 2 (Porter-Thomas):', sp.simplify(F_XEB.subs(M2, 2)))
assert sp.simplify(F_XEB.subs(M2, 2) - a) == 0

# Exact Haar value M2 = 2D/(D+1)
F_haar = sp.simplify(F_XEB.subs(M2, 2 * D / (D + 1)))
print('  at M2 = 2D/(D+1) (exact Haar):', F_haar)
assert sp.simplify(F_haar - a * (D - 1) / (D + 1)) == 0
print('  large-D limit:', sp.limit(F_haar, D, sp.oo))

# Substituting a = (1-eps)^G
print('\nF_XEB =', sp.simplify(F_XEB.subs(M2, 2).subs(a, (1 - eps)**G)))

In [ ]:
# Part (b): the 1/e crossing, exactly and in the small-eps approximation
G_exact = sp.solve(sp.Eq((1 - eps)**G, sp.exp(-1)), G)[0]
print('exact  :  G = ', sp.simplify(G_exact))
print('           = ', sp.simplify(G_exact.subs(eps, sp.Rational(1, 1000))),
      '=', float(G_exact.subs(eps, sp.Rational(1, 1000))))

# small-eps expansion: ln(1-eps) = -eps - eps^2/2 - ...
print('\nseries of -1/ln(1-eps) about eps=0:',
      sp.series(-1 / sp.log(1 - eps), eps, 0, 2))
print('leading order  G ~ 1/eps =', float(1 / sp.Rational(1, 1000)))

---
## Numerical Verification

In [ ]:
import numpy as np

rng = np.random.default_rng(7)

def haar_probs(D, rng):
    '''Output probabilities |<x|psi>|^2 of a Haar-random state on D dimensions.'''
    psi = rng.normal(size=D) + 1j * rng.normal(size=D)
    psi /= np.linalg.norm(psi)
    return np.abs(psi)**2

# The collision term 2^N sum_x P^2: computed vs analytic
print('Collision term  2^N sum_x P_ideal(x)^2\n')
print(f"{'N':>3} {'D':>6} {'computed':>10} {'exact 2D/(D+1)':>16} {'PT limit':>10}")
print('-' * 50)
for N in (4, 6, 8, 10, 12):
    D = 2**N
    vals = [D * np.sum(haar_probs(D, rng)**2) for _ in range(400)]
    exact = 2 * D / (D + 1)
    print(f'{N:>3} {D:>6} {np.mean(vals):>10.4f} {exact:>16.4f} {2.0:>10.4f}')
    assert abs(np.mean(vals) - exact) < 0.02

In [ ]:
# Part (a): F_XEB vs the analytic (1-eps)^G
N, eps_v = 12, 1e-3
D = 2**N
print(f'F_XEB for N={N} qubits, eps={eps_v}\n')
print(f"{'G':>6} {'F_XEB (computed)':>18} {'(1-eps)^G':>12} {'a(D-1)/(D+1)':>14}")
print('-' * 54)
for G_v in (100, 500, 1000, 2000, 4000):
    a_v = (1 - eps_v)**G_v
    vals = []
    for _ in range(200):
        Pi = haar_probs(D, rng)
        Pe = a_v * Pi + (1 - a_v) / D          # global depolarizing model
        vals.append(D * np.sum(Pi * Pe) - 1)   # F_XEB definition
    F = np.mean(vals)
    print(f'{G_v:>6} {F:>18.6f} {a_v:>12.6f} {a_v*(D-1)/(D+1):>14.6f}')
    assert abs(F - a_v) < 0.01

print('\nF_XEB tracks (1-eps)^G, with the exact Haar answer a(D-1)/(D+1).')

In [ ]:
# Part (b): where does F_XEB fall to 1/e?
eps_v = 1e-3
G_exact_v = -1 / np.log(1 - eps_v)
G_approx = 1 / eps_v

print(f'eps = {eps_v}\n')
print(f'  exact   G = -1/ln(1-eps) = {G_exact_v:.2f}')
print(f'  approx  G = 1/eps        = {G_approx:.2f}')
print(f'  relative difference      = {abs(G_exact_v-G_approx)/G_exact_v:.2%}')
print(f'\n  check: (1-eps)^{G_exact_v:.2f} = {(1-eps_v)**G_exact_v:.6f}')
print(f'         1/e                = {np.exp(-1):.6f}')
print(f'  check: (1-eps)^{G_approx:.0f}    = {(1-eps_v)**G_approx:.6f}')

assert abs((1 - eps_v)**G_exact_v - np.exp(-1)) < 1e-9
assert abs((1 - eps_v)**G_approx - np.exp(-1)) < 1e-3
print(f'\n  Both round to G ~ 10^3 gates.')

---
## Visual Pedagogical Proof

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams['figure.dpi'] = 150
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

# Left: Porter-Thomas statistics of the ideal output
N = 12
D = 2**N
P = np.concatenate([haar_probs(D, rng) for _ in range(20)])
x = P * D                                     # rescaled p/mu
ax1.hist(x, bins=80, range=(0, 6), density=True, color='royalblue',
         alpha=0.65, label=f'Haar output, $N={N}$')
tt = np.linspace(0, 6, 200)
ax1.plot(tt, np.exp(-tt), 'crimson', lw=2,
         label=r'Porter--Thomas $e^{-p/\mu}$')
ax1.set_xlabel(r'$p\,/\,\mu$, with $\mu = 2^{-N}$')
ax1.set_ylabel('density')
ax1.set_title(r'Anti-concentration gives $2^N\sum_x P^2 \approx 2$')
ax1.legend()
ax1.grid(True, ls='--', alpha=0.4)

# Right: the XEB decay and the 1/e point
eps_v = 1e-3
Gs = np.linspace(1, 4000, 400)
ax2.plot(Gs, (1 - eps_v)**Gs, 'royalblue', lw=2, label=r'$(1-\varepsilon)^G$')
ax2.plot(Gs, np.exp(-eps_v * Gs), 'k--', lw=1.4, alpha=0.8,
         label=r'$e^{-\varepsilon G}$')

G_pts = np.array([100, 500, 1000, 2000, 4000])
F_pts = []
for G_v in G_pts:
    a_v = (1 - eps_v)**G_v
    Pi = haar_probs(D, rng)
    F_pts.append(D * np.sum(Pi * (a_v * Pi + (1 - a_v) / D)) - 1)
ax2.scatter(G_pts, F_pts, color='crimson', zorder=5, s=35,
            label=r'$F_{\rm XEB}$ (simulated)')

ax2.axhline(np.exp(-1), color='gray', ls=':', lw=1.5)
ax2.axvline(1 / eps_v, color='gray', ls=':', lw=1.5)
ax2.text(1150, 0.72, r'$G \approx 1/\varepsilon = 10^{3}$', fontsize=10,
         color='dimgray')
ax2.text(2600, 0.40, r'$1/e$', fontsize=10, color='dimgray')
ax2.set_xlabel(r'Gate count $G$')
ax2.set_ylabel(r'$F_{\mathrm{XEB}}$')
ax2.set_title(r'XEB decay at $\varepsilon = 10^{-3}$')
ax2.legend()
ax2.grid(True, ls='--', alpha=0.4)

plt.tight_layout()
plt.show()